# SHIPP — Validate Agent `save_item` Boundary

`Gold candidate → current Lakebase status → idempotent saved_items write → agent_activity audit`

**Read-only by default.** The only cell that writes is gated by `RUN_WRITE_TEST`, which stays `False` until every gate above it passes.

## Before you run
1. `psycopg[binary]` must be in this notebook's **Environment** panel (it already is). Do **not** add `%pip` + `restartPython()` after `%run`: the restart wipes everything the bootstrap loaded.
2. Fill in the two widgets at the top: the SHIPP Lakebase **instance** and the **database** that contains schema `shipp`.

## Gates
| Gate | Checks | Writes? |
|---|---|---|
| 0 | bootstrap + agent imports | no |
| 1 | Lakebase connection opens and sees `shipp` | no |
| 2 | unique constraint + `action_status` values match the Agent | no |
| 3 | Gold has every column the Agent reads (needs Gold built) | no |
| 4 | one deterministic Gold candidate exists in Lakebase | no |
| 5 | save → duplicate → rejected, each confirmed by SELECT | **yes, only if RUN_WRITE_TEST** |

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
# Gate 0a — parameters. Widgets keep connection details out of code.
import os, sys, json

dbutils.widgets.text("lakebase_instance", "", "Lakebase instance name")
dbutils.widgets.text("lakebase_database", "", "Database holding schema shipp")
dbutils.widgets.text("lakebase_schema", "shipp", "Lakebase schema")

RUN_WRITE_TEST = False  # SAFETY SWITCH — set True only after Gates 0–4 pass

LAKEBASE_INSTANCE = dbutils.widgets.get("lakebase_instance").strip()
LAKEBASE_DATABASE = dbutils.widgets.get("lakebase_database").strip()
LAKEBASE_SCHEMA = dbutils.widgets.get("lakebase_schema").strip() or "shipp"

assert LAKEBASE_INSTANCE, "BLOCKED: fill the 'lakebase_instance' widget (see Gate 1a below to find it)."
assert LAKEBASE_DATABASE, "BLOCKED: fill the 'lakebase_database' widget (run SELECT current_database() where you query shipp.listings)."

# clients.py reads these; outside a Databricks App nothing else sets them.
os.environ["SHIPP_LAKEBASE_INSTANCE"] = LAKEBASE_INSTANCE
os.environ["SHIPP_LAKEBASE_DATABASE"] = LAKEBASE_DATABASE

print("RUN_WRITE_TEST   =", RUN_WRITE_TEST)
print("Lakebase target  =", LAKEBASE_INSTANCE, "/", LAKEBASE_DATABASE, "/", LAKEBASE_SCHEMA)

In [ ]:
# Gate 0b — agent imports (package lives at agent/agent_ship/src/shipp/agent).
agent_src = os.path.join(REPO_ROOT, "agent", "agent_ship", "src")
if agent_src not in sys.path:
    sys.path.insert(0, agent_src)

from shipp.agent.contracts import ActionStatus, CANDIDATE_MATCH_COLUMNS
from shipp.agent.lakebase import LakebaseRepo
from shipp.agent.config import AgentSettings
from shipp.agent.clients import build_lakebase_connect, build_workspace_client, resolve_pg_target

print("PASS Gate 0 — agent package imported from", agent_src)

In [ ]:
# Gate 1a — confirm the target instance is visible to you. Read-only.
workspace = build_workspace_client()
me = workspace.current_user.me().user_name
print("Databricks user:", me)

visible = {i.name: i for i in workspace.database.list_database_instances()}
print("Instances visible to you:")
for name, inst in visible.items():
    marker = "  <== TARGET" if name == LAKEBASE_INSTANCE else ""
    print(f"  {name} | {inst.read_write_dns} | {inst.state}{marker}")

assert LAKEBASE_INSTANCE in visible, (
    f"BLOCKED: instance '{LAKEBASE_INSTANCE}' is not visible to {me}. "
    "Either the name is wrong, the instance owner has not granted you access, "
    "or SHIPP's database is not a database instance (see the step-by-step notes)."
)

In [ ]:
# Gate 1b — open one connection and prove it is the SHIPP database. Read-only.
settings = AgentSettings(
    llm_endpoint="unused-for-save-validation",
    sql_warehouse_id="unused-for-save-validation",
    candidate_matches_table=GOLD_CANDIDATE_MATCHES,
    search_index=f"{CATALOG}.{GOLD_SCHEMA}.gold_listing_search_docs_index",
    lakebase_schema=LAKEBASE_SCHEMA,
    lakebase_instance=LAKEBASE_INSTANCE,
)
print("Resolved target (host, user, database):", resolve_pg_target(settings, workspace))

connect = build_lakebase_connect(settings, workspace)
lakebase = LakebaseRepo(connect, settings.lakebase_schema)

with connect() as conn, conn.cursor() as cur:
    cur.execute(
        "SELECT current_user, current_database(), "
        "EXISTS (SELECT 1 FROM information_schema.schemata WHERE schema_name = %s)",
        (LAKEBASE_SCHEMA,),
    )
    pg_user, pg_db, has_schema = cur.fetchone()
    print("Connected as:", pg_user, "| database:", pg_db, "| schema exists:", has_schema)
    assert has_schema, f"BLOCKED: schema '{LAKEBASE_SCHEMA}' not found in database '{pg_db}'."

    cur.execute(f"SELECT count(*) FROM {LAKEBASE_SCHEMA}.listings")
    print("Listings in Lakebase:", cur.fetchone()[0])

print("PASS Gate 1 — Lakebase connection works")

In [ ]:
# Gate 2 — real PostgreSQL constraints must match what the Agent writes. Read-only.
def constraints(table):
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            "SELECT conname, pg_get_constraintdef(oid) FROM pg_constraint "
            "WHERE conrelid = %s::regclass ORDER BY conname",
            (f"{LAKEBASE_SCHEMA}.{table}",),
        )
        return dict(cur.fetchall())

saved_constraints = constraints("saved_items")
activity_constraints = constraints("agent_activity")

print("saved_items constraints:")
for n, d in saved_constraints.items():
    print("  ", n, "=>", d)
print("agent_activity constraints:")
for n, d in activity_constraints.items():
    print("  ", n, "=>", d)

# 3a — ON CONFLICT (user_id, request_id, listing_id) needs a UNIQUE on exactly these columns.
# Checked by definition, not by name, so a renamed migration still passes.
def is_save_unique(definition):
    d = definition.upper().replace(" ", "")
    return d.startswith(("UNIQUE(", "PRIMARYKEY(")) and all(
        c in d for c in ("USER_ID", "REQUEST_ID", "LISTING_ID")
    ) and d.count(",") == 2

assert any(is_save_unique(d) for d in saved_constraints.values()), (
    "BLOCKED: no UNIQUE (user_id, request_id, listing_id) on saved_items. "
    "save_item's duplicate protection depends on it."
)

# 3b — every value the Agent writes must be accepted by the CHECK on action_status.
action_status_def = next(
    (d for d in activity_constraints.values() if "action_status" in d), ""
)
agent_values = [s.value for s in ActionStatus]
missing_action_values = [v for v in agent_values if f"'{v}'" not in action_status_def]

print("Agent writes action_status values:", agent_values)
print("DB CHECK definition:", action_status_def or "(no CHECK on action_status)")
print("Values the DB would reject:", missing_action_values or "none")
assert not missing_action_values, (
    "BLOCKED: agent_activity.action_status CHECK rejects values the Agent writes. "
    "Align ActionStatus in contracts.py with the CHECK above (the DB is the operational truth)."
)
print("PASS Gate 2 — constraints match the Agent")

In [ ]:
# Gate 3 — Gold must be consumable by the Agent before any write test.
assert table_exists(GOLD_CANDIDATE_MATCHES), f"BLOCKED: missing {GOLD_CANDIDATE_MATCHES}"
gold = spark.table(GOLD_CANDIDATE_MATCHES)

missing_agent_columns = [c for c in CANDIDATE_MATCH_COLUMNS if c not in gold.columns]
print("Gold columns:", gold.columns)
print("Missing Agent columns:", missing_agent_columns or "none")
assert not missing_agent_columns, (
    "BLOCKED: fix the Gold <-> Agent schema contract before testing save_item. "
    f"Missing: {missing_agent_columns}"
)
print("PASS Gate 3 — Gold matches the Agent contract")

In [ ]:
# Gate 4 — choose one deterministic Gold candidate and confirm it exists in Lakebase. Read-only.
from pyspark.sql import functions as F

candidate = (
    gold.orderBy("request_id", F.col("match_score").desc(), "listing_id")
    .select("request_id", "listing_id", "match_score")
    .first()
)
assert candidate is not None, "BLOCKED: Gold has no candidate rows."

REQUEST_ID = str(candidate["request_id"])
LISTING_ID = str(candidate["listing_id"])

request_state = lakebase.get_request(REQUEST_ID)
listing_state = lakebase.get_listing(LISTING_ID)
assert request_state is not None, f"BLOCKED: Gold request_id {REQUEST_ID} is missing from Lakebase."
assert listing_state is not None, f"BLOCKED: Gold listing_id {LISTING_ID} is missing from Lakebase."

USER_ID = request_state.requester_id  # the save must belong to the request's owner

# Any listing that is NOT available, for the rejection test. None found -> that case is skipped.
with connect() as conn, conn.cursor() as cur:
    cur.execute(
        f"SELECT listing_id, status FROM {LAKEBASE_SCHEMA}.listings "
        "WHERE upper(status) <> 'AVAILABLE' ORDER BY listing_id LIMIT 1"
    )
    unavailable_row = cur.fetchone()
UNAVAILABLE_LISTING_ID = unavailable_row[0] if unavailable_row else None

print("Candidate for write test:")
print("  user_id     :", USER_ID)
print("  request_id  :", REQUEST_ID, "| status:", request_state.status)
print("  listing_id  :", LISTING_ID, "| status:", listing_state.status, "| score:", candidate["match_score"])
print("  unavailable :", unavailable_row or "none found (rejection case will be skipped)")
print("PASS Gate 4 — candidate ready. Review it, then set RUN_WRITE_TEST = True in Gate 0a.")

In [ ]:
# Gate 5 — THE ONLY WRITING CELL. Safe to rerun: a second run sees DUPLICATE, never a second row.
def saved_count(listing_id):
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            f"SELECT count(*) FROM {LAKEBASE_SCHEMA}.saved_items "
            "WHERE user_id = %s AND request_id = %s AND listing_id = %s",
            (USER_ID, REQUEST_ID, listing_id),
        )
        return cur.fetchone()[0]

def recent_audit(listing_id):
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            f"SELECT action_status, created_at FROM {LAKEBASE_SCHEMA}.agent_activity "
            "WHERE user_id = %s AND entity_id = %s AND tool_name = 'save_item' "
            "ORDER BY created_at DESC LIMIT 5",
            (USER_ID, listing_id),
        )
        return cur.fetchall()

if not RUN_WRITE_TEST:
    print("REVIEW MODE — no write executed. Set RUN_WRITE_TEST = True in Gate 0a after Gates 0–4 pass.")
else:
    before = saved_count(LISTING_ID)
    first = lakebase.save_item(user_id=USER_ID, request_id=REQUEST_ID, listing_id=LISTING_ID)
    second = lakebase.save_item(user_id=USER_ID, request_id=REQUEST_ID, listing_id=LISTING_ID)
    after = saved_count(LISTING_ID)

    print("Saved rows before / after :", before, "/", after)
    print("First save  :", first)
    print("Second save :", second)
    print("Audit rows  :", recent_audit(LISTING_ID))

    # 5a — valid save succeeds (or was already saved by an earlier run of this cell)
    assert first.status in (ActionStatus.SUCCESS, ActionStatus.REJECTED_DUPLICATE), first
    # 5b — duplicate is rejected and never creates a second row
    assert second.status is ActionStatus.REJECTED_DUPLICATE, second
    assert after == 1, f"FAILED: expected exactly 1 saved_items row, found {after}"
    # 5c — both attempts are audited
    audit_statuses = {row[0] for row in recent_audit(LISTING_ID)}
    assert ActionStatus.REJECTED_DUPLICATE.value in audit_statuses, "FAILED: duplicate attempt not audited"

    # 5d — unavailable listing is rejected and writes no saved row
    if UNAVAILABLE_LISTING_ID:
        rejected = lakebase.save_item(
            user_id=USER_ID, request_id=REQUEST_ID, listing_id=UNAVAILABLE_LISTING_ID
        )
        print("Unavailable save:", rejected)
        assert rejected.status is ActionStatus.REJECTED_UNAVAILABLE, rejected
        assert saved_count(UNAVAILABLE_LISTING_ID) == 0, "FAILED: unavailable listing was saved"
    else:
        print("SKIPPED 5d — no non-available listing exists. Seed one WITHDRAWN listing to prove it.")

    print("PASS Gate 5 — save_item is validated, idempotent and audited.")

In [ ]:
summary = {
    "stage": "agent_save_item_validation",
    "write_test_enabled": RUN_WRITE_TEST,
    "lakebase": {"instance": LAKEBASE_INSTANCE, "database": LAKEBASE_DATABASE, "schema": LAKEBASE_SCHEMA},
    "gold_contract_ready": not missing_agent_columns,
    "db_action_status_ready": not missing_action_values,
    "candidate": {"user_id": USER_ID, "request_id": REQUEST_ID, "listing_id": LISTING_ID},
    "unavailable_case": "tested" if (RUN_WRITE_TEST and UNAVAILABLE_LISTING_ID) else "not run",
    "status": "PASS" if RUN_WRITE_TEST else "READY_FOR_WRITE_TEST",
}
print(json.dumps(summary, indent=2, default=str))